# 01 · Задача и данные

Модель — ассистент студента, который пишет выпускную работу в редакторе: рядом открыт фрагмент
документа, студент просит, модель отвечает одним сообщением. Инструментов и циклов нет: учим то,
как модель пишет и где проводит границы, это переносится в любую обвязку.

Что должна делать модель:

- содержательные решения оставлять студенту: тему, цель, гипотезу, выводы;
- опираться только на открытый фрагмент, не выдумывать его содержимое;
- правки формы предлагать к принятию, а не переписывать молча;
- отказывать на просьбах обойти проверку, выдумать данные, написать работу за студента;
- заканчивать одним шагом или одним вопросом.

Наборы: `train` для обучения, `dev` для perplexity и preference accuracy, `test_product` — голд-сет
продукта с рубриками и ответами действующего агента, `test_extended` — наш тест на новых работах.
Документы обучения и тестов не пересекаются.

In [ ]:
import sys
sys.path.insert(0, "../..")

from src import data, infer, metrics, report

splits = {name: data.load(name) for name in data.SPLITS}
for name, ds in splits.items():
    print(f"{name:14} {len(ds):4}")
train = splits["train"]

## Формат

Строка — диалог в формате TRL для предпочтений: `prompt` — сообщения с ролями до реплики студента
включительно, `chosen` и `rejected` — по одному ответу ассистента. Системный промпт и документ уже
внутри `prompt`. SFT берёт `prompt` и `chosen`, методы на парах — все три поля, KTO распаривает.

In [ ]:
row = next(r for r in train if len(r["prompt"]) > 2)
print("роли:", [m["role"] for m in row["prompt"]])
data.show(row)

## Что именно учим

Плохой ответ — грамотный текст, нарушающий ровно одно правило. Разница между столбцами и есть
то, чему учим.

In [ ]:
def side_by_side(row, width=60):
    left, right = row["chosen"][0]["content"].split("\n"), row["rejected"][0]["content"].split("\n")
    print(f"{row['id']} · {data.request(row)[:90]}")
    print(f"{'ЭТАЛОН':<{width}} │ ПЛОХОЙ ОТВЕТ")
    print("─" * (2 * width + 3))
    for i in range(max(len(left), len(right))):
        l, r = (left[i] if i < len(left) else ""), (right[i] if i < len(right) else "")
        for j in range(0, max(len(l), len(r), 1), width):
            print(f"{l[j:j + width]:<{width}} │ {r[j:j + width]}")
    print()


by_id = {r["id"]: r for r in train}
for rid in ("TR-TIB-02", "TR-AWR-22", "TR-FT-06"):
    side_by_side(by_id[rid])

## Автопроверки и судья

Механически проверяются форма и границы: конец шагом или вопросом, не больше двух вопросов, нет
готового текста под вставку, опора на документ, отказ где обязателен. Проверка полезна, если проходит
на эталоне и падает на плохом ответе; разница этих долей — её разделяющая способность.
Содержательные пункты рубрики читает LLM-судья, он появится в `02_baseline`.

In [ ]:
rows = [r for ds in splits.values() for r in ds]
passed = {}
for r in rows:
    case = data.case(r)
    for name, ok in metrics.run(r["chosen"][0]["content"], case).items():
        passed.setdefault(name, [[], []])[0].append(ok)
    for name, ok in metrics.run(r["rejected"][0]["content"], case).items():
        passed[name][1].append(ok)

print(f"{'проверка':16}{'назначено':>10}{'эталон':>9}{'плохой':>9}{'разделяет':>11}")
for name, (good, bad) in sorted(passed.items(), key=lambda kv: sum(kv[1][1]) / len(kv[1][1])):
    g, b = sum(good) / len(good), sum(bad) / len(bad)
    print(f"{name:16}{len(good):>10}{g:>9.0%}{b:>9.0%}{g - b:>11.0%}")

## Два теста

Тест продукта: 33 ситуации на одной работе, рубрика и вердикт человека для ответа продового агента,
интервал около ±16 пунктов. Расширенный: 100 наших ситуаций на десяти новых документах, с осями,
которых в тесте продукта нет: ловушки на ложный отказ, пустой документ, второй ход, ошибки в статистике.
Интервал около ±10 пунктов. Он основной, тест продукта — подтверждение.

In [ ]:
from collections import Counter

for name, ds in splits.items():
    n = len(ds)
    print(f"{name:14} {n:4}  отказ обязателен {sum(ds['must_refuse']):3}  многоходовых {sum(len(p) > 2 for p in ds['prompt']):3}"
          f"  без документа {sum(not d for d in ds['document']):3}  длина эталона {sum(len(c[0]['content']) for c in ds['chosen']) // n:4}")

product = splits["test_product"]
print()
print("вердикты человека по продовому агенту:", dict(Counter(r["reference"]["human"] or "не заполнен" for r in product)))
data.show(splits["test_extended"][13])